In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import librosa
from transformers import Wav2Vec2Model
import copy
from huggingface_hub import hf_hub_download

# ==========================================
# 1. CONFIGURATION
# ==========================================
BATCH_SIZE = 8
EPOCHS = 10
LEARNING_RATE = 1e-4
MAX_DURATION_SEC = 4.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {DEVICE}")

# ==========================================
# 2. PSEUDO-SPOOFING DSP FUNCTION
# ==========================================
def apply_pseudo_spoof(wav, sr=16000):
    """
    Simulates AI TTS artifacts by destroying phase information 
    (similar to Griffin-Lim vocoder artifacts) and applying synthetic clipping.
    """
    # 1. Phase distortion (Vocoder artifact simulation)
    D = librosa.stft(wav, n_fft=512, hop_length=128)
    mag = np.abs(D)
    
    # Randomize phase to create that "hollow" or "metallic" AI sound
    random_phase = np.exp(1j * (np.random.rand(*mag.shape) * 2 * np.pi))
    wav_fake = librosa.istft(mag * random_phase, hop_length=128)
    
    # 2. Add high-frequency digital clipping (Mechanical harshness)
    wav_fake = np.clip(wav_fake * 1.2, -1.0, 1.0) 
    
    return wav_fake

# ==========================================
# 3. THE UNIVERSAL DATASET
# ==========================================
class UniversalDataset(Dataset):
    def __init__(self, max_len_sec=4, mode="train"):
        self.max_len = int(16000 * max_len_sec)
        self.mode = mode
        self.samples = [] # (path, label, is_indian)
        
        # --- 1. ASVSpoof 2019 (English: Human & Spoof) ---
        print("Loading ASVspoof 2019...")
        asv_root = "/kaggle/input/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/flac"
        asv_proto = "/kaggle/input/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
        
        if os.path.exists(asv_proto):
            with open(asv_proto, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    utt_id = parts[1]
                    label = 0 if parts[4] == 'bonafide' else 1
                    path = os.path.join(asv_root, f"{utt_id}.flac")
                    self.samples.append((path, label, False)) 
        else:
            print("⚠️ ASVSpoof path not found. Please attach the dataset in Kaggle.")
        
        # --- 2. INDIAN LANGUAGES (Human Only -> We will spoof them) ---
        indian_datasets = {
            "Telugu": "/kaggle/input/combined-dataset", 
            "Tamil": "/kaggle/input/common-voice-corpus-21-09-tamil",
            "Hindi": "/kaggle/input/common-voice-hindi",
            "Malayalam": "/kaggle/input/imasc"
        }
        
        for lang, root_path in indian_datasets.items():
            if os.path.exists(root_path):
                print(f"Loading {lang} from {root_path}...")
                files = self._find_files(root_path)
                files = files[:1500] 
                for f in files:
                    self.samples.append((f, 0, True)) 
            else:
                print(f"⚠️ {lang} path not found: {root_path}")
                
        random.shuffle(self.samples)
        print(f"✅ Total Training Samples: {len(self.samples)}")

    def _find_files(self, directory):
        audio_files = []
        for root, dirs, files in os.walk(directory):
            for file in files:
                if file.lower().endswith(('.flac', '.wav', '.mp3')):
                    audio_files.append(os.path.join(root, file))
        return audio_files

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, is_indian = self.samples[idx]
        
        try:
            wav, _ = librosa.load(path, sr=16000)
            
            # --- THE TRICK: PSEUDO-SPOOFING ---
            if is_indian and self.mode == "train":
                if random.random() > 0.5:
                    wav = apply_pseudo_spoof(wav)
                    label = 1 # We just made a Fake!
            
            # Pad/Truncate
            if len(wav) < self.max_len:
                wav = np.pad(wav, (0, self.max_len - len(wav)))
            else:
                wav = wav[:self.max_len]
                
            return torch.tensor(wav, dtype=torch.float32), label, path
            
        except Exception as e:
            # Robustness: Just try the next file
            return self.__getitem__((idx + 1) % len(self))

# Initialize Dataloader
dataset = UniversalDataset(max_len_sec=MAX_DURATION_SEC, mode="train")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# ==========================================
# 4. MODEL ARCHITECTURE
# ==========================================
class AASIST_Backend(nn.Module):
    def __init__(self, in_channels=128, emb_dim=128):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(3, 1), padding=(1, 0))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(3, 1), padding=(1, 0))
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.attention = nn.Sequential(nn.Linear(128, 64), nn.Tanh(), nn.Linear(64, 1))
        self.fc = nn.Linear(128, emb_dim)
        self.classifier = nn.Linear(emb_dim, 2)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.squeeze(-1).transpose(1, 2)
        w = torch.softmax(self.attention(x), dim=1)
        x = torch.sum(w * x, dim=1)
        x = F.relu(self.fc(x))
        return self.classifier(x)

class VoiceDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained("facebook/mms-300m")
        self.proj = nn.Linear(1024, 128) 
        self.backend = AASIST_Backend()

    def forward(self, wav_input):
        with torch.no_grad():
            outputs = self.backbone(wav_input)
            features = outputs.last_hidden_state
        x = self.proj(features).transpose(1, 2).unsqueeze(-1)
        return self.backend(x)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
print("⏳ Initializing Model...")
model = VoiceDetector().to(DEVICE)

# Freeze Backbone to save GPU memory on Kaggle
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW([
    {'params': model.proj.parameters()},
    {'params': model.backend.parameters()}
], lr=LEARNING_RATE, weight_decay=1e-4)

criterion = nn.CrossEntropyLoss()

print(f"🔥 Starting Training on {len(dataset)} files...")
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (audio_tensors, labels, paths) in enumerate(dataloader):
        audio_tensors, labels = audio_tensors.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(audio_tensors)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * audio_tensors.size(0)
        _, preds = torch.max(logits, 1)
        correct += torch.sum(preds == labels.data)
        total += labels.size(0)
        
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] Batch [{batch_idx}/{len(dataloader)}] - Loss: {loss.item():.4f}")

    epoch_loss = running_loss / total
    epoch_acc = correct.double() / total
    
    print(f"🎯 Epoch {epoch+1} Summary: Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")
    
    # Save a separate file for EVERY epoch
    file_name = f"voice_detection_epoch_{epoch+1}.pth"
    torch.save(model.state_dict(), file_name)
    print(f"💾 Saved epoch checkpoint as '{file_name}'")
    
    # Still keep track of the absolute best one
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), "voice_detection_best_model.pth")
        print("🏆 New absolute best model updated!")

print("✅ Training Complete! You can now download 'voice_detection_universal_model.pth' and deploy it to Hugging Face.")

2026-02-11 18:13:06.034966: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770833586.242674      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770833586.304982      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770833586.855685      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770833586.855745      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770833586.855751      24 computation_placer.cc:177] computation placer alr

🚀 Using device: cuda
Loading ASVspoof 2019...
Loading Telugu from /kaggle/input/combined-dataset...
Loading Tamil from /kaggle/input/common-voice-corpus-21-09-tamil...
Loading Hindi from /kaggle/input/common-voice-hindi...
Loading Malayalam from /kaggle/input/imasc...
✅ Total Training Samples: 31380
⏳ Initializing Model...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

🔥 Starting Training on 31380 files...
Epoch [1/10] Batch [0/3922] - Loss: 0.7613
Epoch [1/10] Batch [10/3922] - Loss: 0.6858
Epoch [1/10] Batch [20/3922] - Loss: 0.6802
Epoch [1/10] Batch [30/3922] - Loss: 0.7136
Epoch [1/10] Batch [40/3922] - Loss: 0.5948
Epoch [1/10] Batch [50/3922] - Loss: 0.4964
Epoch [1/10] Batch [60/3922] - Loss: 0.7238
Epoch [1/10] Batch [70/3922] - Loss: 0.2313
Epoch [1/10] Batch [80/3922] - Loss: 0.5475
Epoch [1/10] Batch [90/3922] - Loss: 0.1947
Epoch [1/10] Batch [100/3922] - Loss: 0.1328
Epoch [1/10] Batch [110/3922] - Loss: 0.1159
Epoch [1/10] Batch [120/3922] - Loss: 0.3387
Epoch [1/10] Batch [130/3922] - Loss: 0.2389
Epoch [1/10] Batch [140/3922] - Loss: 0.6009
Epoch [1/10] Batch [150/3922] - Loss: 0.2988
Epoch [1/10] Batch [160/3922] - Loss: 0.1875
Epoch [1/10] Batch [170/3922] - Loss: 0.2340
Epoch [1/10] Batch [180/3922] - Loss: 0.1771
Epoch [1/10] Batch [190/3922] - Loss: 0.3748
Epoch [1/10] Batch [200/3922] - Loss: 0.3043
Epoch [1/10] Batch [210/3922